<a href="https://colab.research.google.com/github/nagahemaramishetty/log-analysis-anomaly-detection/blob/main/Log_Analysis_Anomaly_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ─── STEP 1: Download & Load NASA HTTP Server Log Dataset ───

# Step 1.1 — Download the raw compressed log file from NASA's public FTP archive
!wget -q "ftp://ita.ee.lbl.gov/traces/NASA_access_log_Jul95.gz" -O NASA_access_log_Jul95.gz
print("Downloading completed")

# Step 1.2 — Decompress the .gz file to extract the raw .log file
!gunzip -f NASA_access_log_Jul95.gz
print("Decompression complete")

# Step 1.3 — Load raw log lines into memory
log_file_path = "NASA_access_log_Jul95"

with open(log_file_path, "r", encoding="utf-8", errors="ignore") as f:
    raw_logs = f.readlines()

# Step 1.4 — Inspect volume and structure
total_lines = len(raw_logs)
print(f"Total log lines loaded: {total_lines:,}")
print(f"\n--- Sample Raw Log Lines (first 5) ---")
for line in raw_logs[:5]:
    print(line.strip())

Decompression complete
Total log lines loaded: 1,891,715

--- Sample Raw Log Lines (first 5) ---
199.72.81.55 - - [01/Jul/1995:00:00:01 -0400] "GET /history/apollo/ HTTP/1.0" 200 6245
unicomp6.unicomp.net - - [01/Jul/1995:00:00:06 -0400] "GET /shuttle/countdown/ HTTP/1.0" 200 3985
199.120.110.21 - - [01/Jul/1995:00:00:09 -0400] "GET /shuttle/missions/sts-73/mission-sts-73.html HTTP/1.0" 200 4085
burger.letters.com - - [01/Jul/1995:00:00:11 -0400] "GET /shuttle/countdown/liftoff.html HTTP/1.0" 304 0
199.120.110.21 - - [01/Jul/1995:00:00:11 -0400] "GET /shuttle/missions/sts-73/sts-73-patch-small.gif HTTP/1.0" 200 4179


In [2]:
# ─── STEP 2: Regex Parsing — Unstructured Logs → Structured DataFrame ───

import re
import pandas as pd

# Step 2.1 — Define Regex pattern with named capture groups matching CLF format
log_pattern = re.compile(
    r'(?P<host>\S+)\s+'          # Client IP or hostname
    r'\S+\s+\S+\s+'              # Ident and auth (usually - -)
    r'\[(?P<timestamp>[^\]]+)\]\s+'  # Timestamp inside brackets
    r'"(?P<method>\S+)\s+'       # HTTP method
    r'(?P<endpoint>\S+)\s+'      # Requested endpoint/URL
    r'(?P<protocol>[^"]+)"\s+'   # HTTP protocol version
    r'(?P<status>\d{3})\s+'      # HTTP status code
    r'(?P<bytes>\S+)'            # Response size in bytes
)

# Step 2.2 — Parse all raw log lines using the Regex pattern
parsed_records = []
parse_failures = 0

for line in raw_logs:
    match = log_pattern.search(line)
    if match:
        parsed_records.append(match.groupdict())
    else:
        parse_failures += 1

# Step 2.3 — Convert parsed records into a structured Pandas DataFrame
df = pd.DataFrame(parsed_records)

# Step 2.4 — Type casting — convert status and bytes to numeric
df['status'] = pd.to_numeric(df['status'], errors='coerce')
df['bytes'] = pd.to_numeric(df['bytes'], errors='coerce').fillna(0).astype(int)

# Step 2.5 — Parse timestamp string into datetime object
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d/%b/%Y:%H:%M:%S %z', errors='coerce')

# Step 2.6 — Report parse accuracy
total = len(raw_logs)
parsed = len(parsed_records)
parse_accuracy = (parsed / total) * 100

print(f"Total lines processed : {total:,}")
print(f"Successfully parsed   : {parsed:,}")
print(f"Parse failures        : {parse_failures:,}")
print(f"Parse accuracy        : {parse_accuracy:.2f}%")
print(f"\n--- DataFrame Shape: {df.shape} ---")
print(f"\n--- DataFrame Schema ---")
print(df.dtypes)
print(f"\n--- Sample Structured Records (first 5) ---")
print(df.head())

Total lines processed : 1,891,715
Successfully parsed   : 1,888,739
Parse failures        : 2,976
Parse accuracy        : 99.84%

--- DataFrame Shape: (1888739, 7) ---

--- DataFrame Schema ---
host                            object
timestamp    datetime64[ns, UTC-04:00]
method                          object
endpoint                        object
protocol                        object
status                           int64
bytes                            int64
dtype: object

--- Sample Structured Records (first 5) ---
                   host                 timestamp method  \
0          199.72.81.55 1995-07-01 00:00:01-04:00    GET   
1  unicomp6.unicomp.net 1995-07-01 00:00:06-04:00    GET   
2        199.120.110.21 1995-07-01 00:00:09-04:00    GET   
3    burger.letters.com 1995-07-01 00:00:11-04:00    GET   
4        199.120.110.21 1995-07-01 00:00:11-04:00    GET   

                                          endpoint  protocol  status  bytes  
0                                 /